# 08 — Selección del ganador y registro

Este es el uso final del holdout: compara una sola vez los dos finalistas de la búsqueda y registra el ganador en MLflow Model Registry con el alias `champion`. No se usan las filas de test de competencia para evaluar desempeño.

Inicia el servidor MLflow (`bash scripts/start_mlflow.sh`) antes de ejecutar. Si el holdout ya se evaluó, el código se detiene para evitar volver a usarlo; consulta las corridas finales existentes en MLflow.

In [1]:
from pathlib import Path
import os
import sys

import pandas as pd
import mlflow
from IPython.display import display
from mlflow import MlflowClient

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
if not (ROOT / "src" / "datos.py").exists():
    raise FileNotFoundError("Abre este notebook desde la carpeta PSet3.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.datos import load_obt
from src.preparacion import split_churn_data
from src.seleccion import (
    CHAMPION_ALIAS,
    CHALLENGER_ALIAS,
    FINAL_EXPERIMENT,
    REGISTERED_MODEL_NAME,
    evaluate_finalists_and_register_champion,
)

TRACKING_URI = os.getenv("MLFLOW_TRACKING_URI", "http://127.0.0.1:5001")

In [2]:
mlflow.set_tracking_uri(TRACKING_URI)
client = MlflowClient()
final_experiment = client.get_experiment_by_name(FINAL_EXPERIMENT)

if final_experiment and final_experiment.tags.get("holdout_evaluation_started") == "true":
    print("El holdout ya fue consumido; se muestran las corridas guardadas sin volver a evaluarlo.")
    saved_runs = mlflow.search_runs(experiment_ids=[final_experiment.experiment_id])
    if "params.selection_run_id" in saved_runs:
        saved_runs = saved_runs.loc[saved_runs["params.selection_run_id"].isna()]
    columns = [
        "tags.mlflow.runName",
        "params.model_family",
        "params.selected_as_champion",
        "metrics.holdout_roc_auc",
        "metrics.holdout_average_precision",
        "metrics.holdout_f1",
        "metrics.holdout_precision",
        "metrics.holdout_recall",
        "metrics.holdout_balanced_accuracy",
        "status",
    ]
    display(saved_runs[[column for column in columns if column in saved_runs]])
else:
    obt = load_obt()
    obt.columns = obt.columns.str.lower()
    split = split_churn_data(obt)
    final_results = evaluate_finalists_and_register_champion(split, tracking_uri=TRACKING_URI)
    display(final_results.round(4))

2026/11/26 23:18:01 INFO mlflow.tracking.fluent: Experiment with name 'pset3-churn-final-holdout' does not exist. Creating a new experiment.
/Users/eliaspogo/USFQ/DATA SCIENCE/PSets/PSet3/.venv/lib/python3.12/site-packages/mlflow/types/utils.py:440: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details.
  warnings.warn(
2026/11/26 23:18:01 WARNING mlflow.models.model: `artifact_path` 

🏃 View run gradient_boosting_holdout_final at: http://127.0.0.1:5001/#/experiments/7/runs/503b322b413046a88cca48b13afd0367
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/7
🏃 View run random_forest_holdout_final at: http://127.0.0.1:5001/#/experiments/7/runs/b630b671a9c4440692bf719571389b5e
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/7


2026/11/26 23:18:02 WARNING mlflow.utils.environment: Failed to resolve installed pip version. ``pip`` will be added to conda.yaml environment spec without a version specifier.
Registered model 'pset3-churn-classifier' already exists. Creating a new version of this model...
2026/11/26 23:18:02 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: pset3-churn-classifier, version 4


🏃 View run gradient_boosting_champion_registration at: http://127.0.0.1:5001/#/experiments/7/runs/fc5fd00b2720450f80ce9c7dedd0da7b
🧪 View experiment at: http://127.0.0.1:5001/#/experiments/7


Created version '4' of model 'pset3-churn-classifier'.


,model,best_cv_roc_auc,best_params,holdout_roc_auc,holdout_average_precision,holdout_f1,holdout_precision,holdout_recall,holdout_balanced_accuracy,champion,registry_model,registry_version,registry_alias
0,gradient_boosting,0.9317,"{""model__learning_rate"": 0.07, ""model__max_dep...",0.9323,0.8301,0.7258,0.7868,0.6737,0.8131,True,pset3-churn-classifier,4,champion
1,random_forest,0.9238,"{""model__max_depth"": 15, ""model__min_samples_l...",0.9239,0.8028,0.7169,0.6608,0.7835,0.8395,False,NaN,NaN,NaN


In [3]:
mlflow.set_tracking_uri(TRACKING_URI)
client = MlflowClient()
champion = client.get_model_version_by_alias(
    REGISTERED_MODEL_NAME,
    CHAMPION_ALIAS,
)
challenger = client.get_model_version_by_alias(
    REGISTERED_MODEL_NAME,
    CHALLENGER_ALIAS,
)
print(
    f"Champion: {REGISTERED_MODEL_NAME}, "
    f"versión {champion.version}, run_id={champion.run_id}"
)
print(
    f"Challenger: {REGISTERED_MODEL_NAME}, "
    f"versión {challenger.version}, run_id={challenger.run_id}"
)

Champion: pset3-churn-classifier, versión 4, run_id=fc5fd00b2720450f80ce9c7dedd0da7b
Challenger: pset3-churn-classifier, versión 3, run_id=15d2b1befb6d42e598aa0b6d96a87fd5


El alias `champion` resuelve a Gradient Boosting y `challenger` a Random Forest. El challenger se registra con sus mejores hiperparámetros de CV, entrenado con desarrollo y sin volver a usar el holdout.